# Compare Proposed Method vs Vanilla — QED Validation

Loads each OPT-350M checkpoint from `saved_model/`, runs greedy generation on the QED validation set,
records wall-clock generation time per batch, and saves per-model JSON files that are then merged into one.

In [13]:
import os
import json
import time
import torch
from pathlib import Path
from tqdm import tqdm
from itertools import chain
from datasets import load_from_disk
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM

In [14]:
# ── Paths ──────────────────────────────────────────────────────────────────────
TOKENIZER_PATH = r"C:/Users/dmin/HUST/Vast.ai_work/Multi-Level-OT/EleutherAI/opt-350m"
DATASET_PATH   = r"C:/Users/dmin/HUST/Vast.ai_work/Multi-Level-OT/llm_distillation/datasets/hf/qedllama3"
SPLIT          = "validation"

MODELS = {
    "proposed_method": r"C:/Users/dmin/HUST/Vast.ai_work/saved_model/proposed_method/seed 26",
    "vanilla":         r"C:/Users/dmin/HUST/Vast.ai_work/saved_model/vanilla/seed 26",
}

OUTPUT_DIR = r"C:/Users/dmin/HUST/Vast.ai_work/Multi-Level-OT/compare_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE     = 8
MAX_NEW_TOKENS = 150
CONTEXT_MAX_LEN = 1024 
NUM_WORKERS    = 0      

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [15]:
# ── Tokenizer ──────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH, trust_remote_code=True)
tokenizer.add_special_tokens({"pad_token": tokenizer.eos_token})
tokenizer.padding_side = "left"   # required for correct batch generation with causal LMs
print(f"Vocab size: {len(tokenizer)}")

Vocab size: 50265


In [16]:
# ── Dataset & prompts ──────────────────────────────────────────────────────────
dataset = load_from_disk(DATASET_PATH)[SPLIT]
print(f"Loaded {len(dataset)} examples from split='{SPLIT}'")
print("Columns:", dataset.column_names)

def build_prompt(item):
    return f"Context: {item['paragraph_text']}\nQuestion: {item['question']}\nAnswer:"

dataset = dataset.map(lambda item: {"prompt": build_prompt(item)})

def tokenise_batch(batch):
    enc = tokenizer(batch["prompt"], padding="longest", truncation=False)
    return enc

dataset_tok = dataset.map(tokenise_batch, batched=True, batch_size=BATCH_SIZE)
before = len(dataset_tok)
dataset_tok = dataset_tok.filter(lambda item: len(item["input_ids"]) <= CONTEXT_MAX_LEN)
print(f"Kept {len(dataset_tok)}/{before} examples (context_max_len={CONTEXT_MAX_LEN})")

# Extract as plain Python lists BEFORE set_format restricts visible columns
ground_truths = list(dataset_tok["answers"])
example_ids   = list(dataset_tok["example_id"])
questions     = list(dataset_tok["question"])

dataset_tok.set_format(type="torch", columns=["input_ids", "attention_mask"])
dataloader = DataLoader(dataset_tok, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
print(f"Batches: {len(dataloader)}")

Loaded 1355 examples from split='validation'
Columns: ['example_id', 'title_text', 'url', 'question', 'paragraph_text', 'sentence_starts', 'original_nq_answers', 'annotation', 'llama3_8b_chat_answers', 'answers']
Kept 1347/1355 examples (context_max_len=1024)
Batches: 169


In [17]:
# ── Inference helper ───────────────────────────────────────────────────────────
def run_inference(model_name: str, model_path: str) -> list[dict]:
    """
    Load model, run batched greedy generation on the dataloader,
    return a list of per-example dicts with prediction and timing.
    """
    print(f"\n{'='*60}")
    print(f"Loading model: {model_name}")
    print(f"  from: {model_path}")

    model = AutoModelForCausalLM.from_pretrained(model_path).to(device)
    model.resize_token_embeddings(len(tokenizer))   # consistent with training setup
    model.config.pad_token_id = tokenizer.pad_token_id
    model.eval()
    print(f"Model loaded ({sum(p.numel() for p in model.parameters())/1e6:.1f}M params)")

    results = []
    example_ptr = 0   # tracks which example we're on across batches

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=model_name):
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            prompt_len     = input_ids.shape[1]   # same for all items (left-padded)

            t0 = time.perf_counter()
            output = model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
            )
            t1 = time.perf_counter()
            batch_time_s = t1 - t0

            # Slice off the prompt; decode generated tokens only
            new_tokens = output[:, prompt_len:]
            sentences  = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

            bsz = len(sentences)
            for i, pred in enumerate(sentences):
                # Take only the first line (stop at any newline)
                pred = pred.split("\n")[0].strip()
                idx  = example_ptr + i
                results.append({
                    "example_id":         int(example_ids[idx]),
                    "question":           questions[idx],
                    "ground_truth":       ground_truths[idx],
                    "prediction":         pred,
                    "generation_time_s":  round(batch_time_s, 4),  # shared batch time
                    "batch_size":         bsz,
                })
            example_ptr += bsz

    del model
    torch.cuda.empty_cache()
    print(f"Done. Collected {len(results)} predictions.")
    return results

In [18]:
# ── Run inference for each model ───────────────────────────────────────────────
all_results: dict[str, list[dict]] = {}

for model_name, model_path in MODELS.items():
    preds = run_inference(model_name, model_path)
    all_results[model_name] = preds

    out_path = os.path.join(OUTPUT_DIR, f"{model_name}_predictions.json")
    with open(out_path, "w") as f:
        json.dump(preds, f, indent=2)
    print(f"Saved → {out_path}")


Loading model: proposed_method
  from: C:/Users/dmin/HUST/Vast.ai_work/saved_model/proposed_method/seed 26


Loading weights: 100%|██████████| 388/388 [00:00<00:00, 8932.08it/s]


Model loaded (331.2M params)


proposed_method: 100%|██████████| 169/169 [01:08<00:00,  2.46it/s]


Done. Collected 1347 predictions.
Saved → C:/Users/dmin/HUST/Vast.ai_work/Multi-Level-OT/compare_outputs\proposed_method_predictions.json

Loading model: vanilla
  from: C:/Users/dmin/HUST/Vast.ai_work/saved_model/vanilla/seed 26


Loading weights: 100%|██████████| 388/388 [00:00<00:00, 5642.90it/s]


Model loaded (331.2M params)


vanilla: 100%|██████████| 169/169 [01:10<00:00,  2.39it/s]


Done. Collected 1347 predictions.
Saved → C:/Users/dmin/HUST/Vast.ai_work/Multi-Level-OT/compare_outputs\vanilla_predictions.json


In [19]:
# ── Merge both models into a single JSON ───────────────────────────────────────
# Index each model's results by example_id for O(1) lookup
def index_by_id(results: list[dict]) -> dict[int, dict]:
    return {r["example_id"]: r for r in results}

proposed_idx = index_by_id(all_results["proposed_method"])
vanilla_idx  = index_by_id(all_results["vanilla"])

# Build merged records over the shared set of example_ids
shared_ids = sorted(set(proposed_idx) & set(vanilla_idx))
print(f"Merging {len(shared_ids)} examples present in both models.")

merged = []
for eid in shared_ids:
    p = proposed_idx[eid]
    v = vanilla_idx[eid]
    assert p["ground_truth"] == v["ground_truth"], \
        f"Ground truth mismatch at example_id={eid}"
    merged.append({
        "example_id":   eid,
        "question":     p["question"],
        "ground_truth": p["ground_truth"],
        "proposed_method": {
            "prediction":        p["prediction"],
            "generation_time_s": p["generation_time_s"],
            "batch_size":        p["batch_size"],
        },
        "vanilla": {
            "prediction":        v["prediction"],
            "generation_time_s": v["generation_time_s"],
            "batch_size":        v["batch_size"],
        },
    })

merged_path = os.path.join(OUTPUT_DIR, "merged_predictions.json")
with open(merged_path, "w") as f:
    json.dump(merged, f, indent=2)
print(f"Merged file saved → {merged_path}")

Merging 1347 examples present in both models.
Merged file saved → C:/Users/dmin/HUST/Vast.ai_work/Multi-Level-OT/compare_outputs\merged_predictions.json


In [20]:
for record in merged[:3]:
    print(f"Q : {record['question']}")
    print(f"GT: {record['ground_truth']}")
    print(f"PM: {record['proposed_method']['prediction']}  ({record['proposed_method']['generation_time_s']:.3f}s batch)")
    print(f"VN: {record['vanilla']['prediction']}  ({record['vanilla']['generation_time_s']:.3f}s batch)")
    print("-" * 70)

Q : what is billy last name in where the red fern grows
GT: Colman
PM: Colman  (0.304s batch)
VN: Colman  (0.302s batch)
----------------------------------------------------------------------
Q : how long is the bridge between new brunswick and prince edward island
GT: 12.9 - kilometre ( 8 mi )
PM: 12.9 - kilometre ( 8 mi )  (0.525s batch)
VN: 12.9 - kilometre ( 8 mi )  (0.502s batch)
----------------------------------------------------------------------
Q : who is recognized as the founder of islam
GT: Muhammad
PM: Muhammad  (0.361s batch)
VN: Muhammad  (0.477s batch)
----------------------------------------------------------------------
